##### Setup Environment

%pip install langchain_community langchainhub langchain langchain-chroma chromadb langchain-openai

%pip install sentence-transformers sentence-transformers

In [ ]:
import os

os.getenv('OPENAI_API_KEY')
if "OPENAI_API_KEY" not in os.environ:
    print("Setup OPENAI_API_KEY in Environment Vars")

if "OPENAI_API_BASE" not in os.environ:
    print("Setup OPENAI_API_BASE in Environment Vars")

##### Load Document

In [2]:
from com.example.agentic.loader.LoadManager import LoadManager
#
docs = LoadManager.from_web(['https://educosys.com/#faq'])

#
print(docs)

USER_AGENT environment variable not set, consider setting it to identify your requests.


[DEBUG] Found 1 page : ['https://educosys.com/#faq']
[DEBUG] Loading pages from : https://educosys.com/#faq
[DEBUG] Loaded 1 pages from https://educosys.com/#faq
[Document(metadata={'source': 'https://educosys.com/#faq', 'title': 'Educosys', 'description': 'Unlock Your Potential with Keerti Purswani. Join Anytime. Get Lifetime Access to All Past, Ongoing and Future Batches', 'language': 'en'}, page_content='Educosys')]


In [ ]:
from langchain_text_splitters  import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
splits = text_splitter.split_documents(docs)

print(len(splits))
for index, split in enumerate(splits):
    print(f"{index} : {split}")


1
0 : page_content='Educosys' metadata={'source': 'https://educosys.com/#faq', 'title': 'Educosys', 'description': 'Unlock Your Potential with Keerti Purswani. Join Anytime. Get Lifetime Access to All Past, Ongoing and Future Batches', 'language': 'en'}


: 

##### Add Docs to Vactore DB

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
import openai
from langchain_huggingface import HuggingFaceEmbeddings


# Using HuggingFace for embeddings, storing in Chroma
# huggingFaceEmbeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
huggingFaceEmbeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

#
# openAIembeddings = OpenAIEmbeddings()
# openAIembeddings = OpenAIEmbeddings(
#    model="text-embedding-3-large",
#    # With the `text-embedding-3` class
#    # of models, you can specify the size
#    # of the embeddings you want returned.
#    # dimensions=1024
# )

vector_store = Chroma.from_documents(
    documents=splits, 
    embedding=huggingFaceEmbeddings 
)

print(vector_store._collection.count())
print(vector_store._collection.get())

40
{'ids': ['99b50a1c-2aac-4765-8ee3-f93544f51b33', '18d8dc2d-8521-475d-93f9-219898900f45', '4396f3c0-31cc-4f2b-9b62-e61f7c38a30d', '95cf053a-cecf-4c52-b4b2-e8e617c81fa9', '41a62509-dcde-4eb7-acd3-4551f3a8c969', 'a6b11f48-25e0-4081-909e-8f797268d520', '959402b0-83b0-47a5-9774-14a7e9bcbc50', '4db284ae-6f5f-47cf-94dc-439a1cfcc363', '561694e5-957e-4a65-8e6f-99b7505f6853', '728e3228-c660-46f3-bb07-4e27b9b94f6f', '059717a1-be71-4f9c-add4-5ed9b27e9446', 'db2e0572-cf44-4125-a792-daa486faaa86', 'e6ddb974-c579-4409-a6e3-d36724b7ee45', '3e50434c-3922-4bb9-a66e-51a39cd54cfa', 'ec7c34c0-5d42-4146-8c02-3afec722b833', '7860c11e-a298-4d83-a1d4-54df22205d10', '8f471e7c-be9c-4fb1-a617-c5c482adad1c', '0388066f-1174-4dcb-8532-4cfb15929249', '198aa6fe-9a9b-48a2-9f97-5b5ee5cc41b6', '7a47d83e-834d-4a41-8acc-553b464c1b72', '99c3f055-46ea-469d-bb0f-32916d3b6c34', '7b463b9e-2fa3-4bce-80b9-b6bacaf5ff5d', 'e7838768-15f8-4128-b9cc-6d7aa5a7c7f8', 'ded002c6-54de-485d-8436-9aef2626e67c', '0c2b15b2-f95c-4d42-a9ea-276

##### RAG Pipeline

In [6]:

retriever = vector_store.as_retriever()

In [9]:
#from langchain import hub
# Hub
from langchain_classic import hub

prompt = hub.pull("rlm/rag-prompt")

In [10]:
import os
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="openai/gpt-oss-20b", 
    base_url="https://api.groq.com/openai/v1",
    api_key=os.getenv("OPENAI_API_KEY")
)

In [11]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [12]:
def format_docs(docs):
  return "\n".join(doc.page_content for doc in docs)

In [13]:
rag_chain = ({"context" : retriever | format_docs, "question" : RunnablePassthrough()}
             | prompt
             | llm
             | StrOutputParser())

In [14]:
rag_chain.invoke("Are the recordings of the course available? For how long?")

'Yes, the live sessions are recorded and you can re‑watch them. The recordings are kept available for the entire duration of the 7‑week course (i.e., while you’re enrolled). Once the course ends, the recordings are typically no longer accessible.'

In [15]:
rag_chain.invoke("Are the testimonials for the course available? Name the studenst who have shared testimonials")

'Yes, testimonials are available on the site. The students who have shared them are Ruthira\u202fSekar, Abhijit\u202fMone, and\u202fManika\u202fKaushik.'

In [16]:
rag_chain.invoke("Are the certificates for the course provided?")

'Yes, certificates are provided for the course. This is mentioned in the FAQ section of the course page. The certificate is awarded upon successful completion of the program.'

In [17]:
rag_chain.invoke("What all projects are covered in the course?")

'I’m not sure about the specific projects covered in the course.'

#####
langchain_classis old package supported till DEC-2026

In [ ]:
from langchain_classic.chains import RetrievalQA


qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True
)


In [ ]:
qa_chain.invoke("Are the recordings of the course available? For how long?")

In [ ]:
qa_chain.invoke("Are the testimonials for the course available? Name the studenst who have shared testimonials")

In [ ]:
qa_chain.invoke("Are the certificates for the course provided?")

In [ ]:
qa_chain.invoke("What all projects are covered in the course?")

In [ ]:
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import (
    create_stuff_documents_chain,
)
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI


system_prompt = (
    "Use the given context to answer the question. "
    "If you don't know the answer, say you don't know. "
    "Use three sentence maximum and keep the answer concise. "
    "Context: {context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)
question_answer_chain = create_stuff_documents_chain(llm, prompt)

chain = create_retrieval_chain(retriever, question_answer_chain)
chain.invoke({"input": query})

In [ ]:
from langchain_core.runnables import RunnableLambda

In [ ]:
def print_prompt(prompt_text):
  print("Prompt - ", prompt_text)
  return prompt_text

In [ ]:
rag_chain_with_print = ({"context" : retriever | format_docs, "question" : RunnablePassthrough()}
             | prompt
             | RunnableLambda(print_prompt)
             | llm
             | StrOutputParser())

In [ ]:
rag_chain_with_print.invoke("What all projects are covered in the course?")